# Задание 1. MLM
Используем предобученную модель BERT для задачи MLM. Поработаем с предложением `"Paris is the capital of [MASK]."` и предскажем слово, которое должно стоять вместо `[MASK]`.

Используем `transformers.BertForMaskedLM`. 

Что уже сделано в прекоде:
- Загружены токенизатор и модель bert-base-uncased.
- Задано тестовое предложение с маской.
- Токенизировано предложение с преобразованием в тензоры PyTorch.
- Найдена позиция маски в последовательности токенов.

Ваша задача — дописать недостающие части кода:
- Получить предсказания модели для входных данных.
- Извлечь логиты (сырые оценки) для позиции маски.
- Выбрать токен с наивысшей вероятностью.
- Декодировать и вывести результат.

Выполните задание локально, используйте ВМ, выданную в прошлом уроке. Затем сверьтесь с авторским решением. 

In [ ]:
import torch
from transformers import BertTokenizer, BertForMaskedLM
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', ignore_mismatches=True)
model = BertForMaskedLM.from_pretrained('bert-base-uncased', ignore_mismatched_sizes=True)

sentence = "Paris is the capital of [MASK]."
inputs = tokenizer(sentence, return_tensors="pt")
# Используйте позицию [MASK] и получите предсказание
mask_index = torch.where(inputs["input_ids"][0] == tokenizer.mask_token_id)[0].item()

# # Выполните предсказание и получите логиты
with torch.no_grad():
    output = model(**tokenizer(sentence, return_tensors='pt'))

# Выберите токен с наивысшим логитом в позиции маски с помощью argmax и item
predicted_index = output.logits[0, mask_index, :].argmax().item()
print(tokenizer.decode(predicted_index))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3914.22it/s]

france


# Задание 2. NSP
Решите задачу NSP — проверьте, является ли второе предложение продолжением первого. 
Используйте `BertForNextSentencePrediction` для инициализации модели и проверки, является ли второе предложение продолжением первого.

В `text_a` используйте предложение `The cat sat on the mat`, для `text_b` используйте `It was very sleepy`.

Токенизируйте текст, получите логиты (уже в прекоде) и вероятность с помощью `Softmax`, проверьте, является ли второе предложение продолжением первого.

In [87]:
import torch
from transformers import BertTokenizer, BertForNextSentencePrediction
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()


# Инициализация модели и токенизатора
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', ignore_mismatches=True)
model = BertForNextSentencePrediction.from_pretrained('bert-base-uncased', ignore_mismatched_sizes=True)


# Пример для IsNext (последовательные предложения)
text_a = "The cat sat on the mat"
text_b = "It was very sleepy"

# Токенизация и подготовка входа
inputs = tokenizer(text_a, text_b, return_tensors='pt')

# Получение предсказаний
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Интерпретация результатов
is_next_prob = torch.softmax(logits, dim=1)[0][1].item() # Вероятность IsNext
not_next_prob = torch.softmax(logits, dim=1)[0][0].item()  # Вероятность NotNext

print(f"Вероятность IsNext: {not_next_prob:.4f}")
print(f"Вероятность NotNext: {is_next_prob:.4f}")
print("\n" + "="*80 + "\n")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3831.28it/s]

Вероятность IsNext: 0.9999
Вероятность NotNext: 0.0001




# Задание 3. NSP с новым примером
Решите задачу NSP с другим примером. В `text_a` используйте предложение `The cat sat on the mat`, для `text_b` используйте `The Eiffel Tower is located in Paris`.

Проверьте, является ли второе предложение продолжением первого.

In [88]:
text_a = "The cat sat on the mat"
text_b = "The Eiffel Tower is located in Paris"

# Токенизация и подготовка входа
inputs = tokenizer(text_a, text_b, return_tensors='pt')

# Получение предсказаний
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Интерпретация результатов
is_next_prob = torch.softmax(logits, dim=1)[0][1].item() # Вероятность IsNext
not_next_prob = torch.softmax(logits, dim=1)[0][0].item()  # Вероятность NotNext

print(f"Вероятность IsNext: {not_next_prob:.4f}")
print(f"Вероятность NotNext: {is_next_prob:.4f}")
print("\n" + "="*80 + "\n")

Вероятность IsNext: 0.0002
Вероятность NotNext: 0.9998


